# Visualize Judge Scores

This notebook loads the judge scores from the Assistant Axis pipeline and visualizes the differences between the English and Japanese scores

## Set Up Code

In [ ]:
# Install / import
!pip install --upgrade "kaleido==0.1.*"

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Custom function to save plots
def save_fig(fig, name, scale=3):
    fig.write_image(f"{name}.png", width=fig.layout.width, height=fig.layout.height, scale=scale)

## Load Assistant Axis

In [ ]:
!git clone https://github.com/Arcee183/SAIN_Utrecht_Summer_Challenge.git

%cd SAIN_Utrecht_Summer_Challenge
ROOT_DIR = Path.cwd()
OUT_DIR = ROOT_DIR / "data" / "outputs" / "qwen3-32B"
EN_DIR = OUT_DIR / "English" / "scores"
JA_DIR = OUT_DIR / "Japanese" / "Japanese_prompt" / "scores"

Cloning into 'SAIN_Utrecht_Summer_Challenge'...
remote: Enumerating objects: 1438, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 1438 (delta 53), reused 131 (delta 52), pack-reused 1306 (from 1)
Receiving objects: 100% (1438/1438), 3.22 GiB | 24.07 MiB/s, done.
Resolving deltas: 100% (291/291), done.
Updating files: 100% (492/492), done.
/content/SAIN_Utrecht_Summer_Challenge


In [ ]:
# Folder where plotly figures get exported as PNG (kaleido backend)
PLOT_OUTPUT_DIR = "plots"
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

## Load judge scores


In [ ]:
# Load English role scores
ja_role_scores = {}
for p in Path(JA_DIR).glob("*.json"):
    with open(p, "r") as f:
         ja_role_scores[p.stem] = json.load(f)

print(f"Loaded {len(ja_role_scores)} japanese role scores")

Loaded 55 japanese role scores


In [ ]:
# Load Japanese role scores
en_role_scores = {}
for p in Path(EN_DIR).glob("*.json"):
    with open(p, "r") as f:
         en_role_scores[p.stem] = json.load(f)

print(f"Loaded {len(en_role_scores)} english role scores")

Loaded 55 english role scores


In [ ]:
# Sanity check: filenames should match (same role set) across languages
common = set(en_role_scores) & set(ja_role_scores)
only_en = set(en_role_scores) - set(ja_role_scores)
only_ja = set(ja_role_scores) - set(en_role_scores)
print(f"Matched roles: {len(common)}")
if only_en:
    print("EN-only files (check naming):", only_en)
if only_ja:
    print("JA-only files (check naming):", only_ja)

Matched roles: 55


## Plot Judge scores of English and Japanese responses and compare

In [ ]:
# Okabe-Ito palette (colorblind-safe)
OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442",
             "#0072B2", "#D55E00", "#CC79A7", "#000000"]

In [ ]:
# Parse into a long-format dataframe
KEY_PATTERN = re.compile(r"pos_p(\d+)_q(\d+)")

def build_long_df(scores_dict, language):
    rows = []
    for role, data_dict in scores_dict.items():
        for key, score in data_dict.items():
            m = KEY_PATTERN.match(key)
            if not m:
                continue  # skip unexpected keys rather than crash
            prompt_idx, question_idx = int(m.group(1)), int(m.group(2))
            rows.append({
                "role": role,
                "language": language,
                "prompt": prompt_idx,
                "question": question_idx,
                "score": score,
            })
    return pd.DataFrame(rows)

In [ ]:
df_en = build_long_df(en_role_scores, "EN")
df_ja = build_long_df(ja_role_scores, "JA")
df = pd.concat([df_en, df_ja], ignore_index=True)

print(df.shape)  # expect 55 roles * 250 * 2 languages = 27,500 rows (once all files fetched)
df.head()

(27500, 5)


,role,language,prompt,question,score
0,competitor,EN,0,0,2
1,competitor,EN,0,1,3
2,competitor,EN,0,2,3
3,competitor,EN,0,3,3
4,competitor,EN,0,4,3


In [ ]:
# Per-role summary stats (EN vs JA)
summary = (
    df.groupby(["role", "language"])["score"]
    .agg(mean="mean", std="std", n="count")
    .reset_index()
)

summary, summary.shape

(           role language   mean       std    n
 0    accountant       EN  2.564  0.779960  250
 1    accountant       JA  1.652  0.950003  250
 2         actor       EN  2.916  0.376178  250
 3         actor       JA  1.636  0.931113  250
 4        addict       EN  2.552  0.830696  250
 ..          ...      ...    ...       ...  ...
 105       whale       JA  2.780  0.555730  250
 106       widow       EN  2.840  0.513278  250
 107       widow       JA  2.280  1.006804  250
 108   zeitgeist       EN  2.592  0.695320  250
 109   zeitgeist       JA  1.228  0.627368  250
 
 [110 rows x 5 columns],
 (110, 5))

In [ ]:
pivot_mean = summary.pivot(index="role", columns="language", values="mean")
pivot_mean["delta"] = pivot_mean["EN"] - pivot_mean["JA"]
pivot_mean = pivot_mean.sort_values("delta")
roles_sorted = pivot_mean.index

In [ ]:
pivot_mean

language,EN,JA,delta
role,,,
optimist,2.684,2.748,-0.064
presenter,2.956,2.968,-0.012
teacher,2.932,2.932,0.000
leviathan,2.968,2.964,0.004
pirate,2.972,2.872,0.100
demon,2.920,2.816,0.104
traditionalist,2.880,2.772,0.108
bard,3.000,2.868,0.132
idealist,2.800,2.660,0.140


In [ ]:
count_df = (
    df.groupby(["language", "score"])
    .size()
    .reset_index(name="count")
)
count_df["proportion"] = count_df.groupby("language")["count"].transform(lambda x: x / x.sum())
count_df["score"] = count_df["score"].astype(str)  # treat as categorical for clean x-axis

In [ ]:
# PLOT 1 — Delta bar chart (EN - JA), sorted
delta_df = pivot_mean.reset_index()  # columns: role, EN, JA, delta
delta_df["direction"] = np.where(delta_df["delta"] < 0, "JA higher", "EN higher")
new_df = pd.concat([delta_df.head(10), delta_df.tail(10)])

fig1 = px.bar(
    new_df,
    x="delta",
    y="role",
    orientation="h",
    color="direction",
    color_discrete_map={"EN higher": OKABE_ITO[4], "JA higher": OKABE_ITO[5]},
    labels={"delta": "EN mean - JA mean", "role": "Role"},
    title="Language gap in role-play score, per role (EN - JA)",
)
fig1.update_layout(
    height=max(600, 20 * len(new_df)),  # scale height to number of roles
    width=900,
    # yaxis=dict(categoryorder="array", categoryarray=roles_sorted),
    template="plotly_white",
)
fig1.add_vline(x=0, line_width=1, line_color="black")
fig1.show()

In [ ]:
save_fig(fig1, f"{PLOT_OUTPUT_DIR}/delta_en_ja.png")

In [ ]:
# PLOT 2 — Overall score distribution, EN vs JA (pooled)
fig2 = px.bar(
    count_df,
    x="score",
    y="proportion",
    color="language",
    barmode="group",
    color_discrete_map={"EN": OKABE_ITO[4], "JA": OKABE_ITO[5]},
    labels={
        "score": "Score (0=no response, 1=no role, 2=somewhat role, 3=fully role)",
        "proportion": "Proportion of all scores",
        "language": "Language",
    },
    title="Overall score distribution: EN vs JA (all roles pooled)",
)
fig2.update_layout(width=800, height=550, template="plotly_white")
fig2.show()

In [ ]:
save_fig(fig2, f"{PLOT_OUTPUT_DIR}/distribution_en_ja.png")